# MJ AI — first fine-tuning run
This notebook trains an MJ AI LoRA adapter on top of Qwen2.5-0.5B-Instruct. It does not train a foundation model from zero. Use a GPU runtime if available, and save the output before the session ends.

The bundled seed dataset is only for testing the pipeline. For useful behavior, upload a larger, high-quality JSONL dataset you own or have permission to use.

In [ ]:
!pip -q install transformers datasets peft accelerate
!git clone https://github.com/singwithsashu-a11y/Ai-video-free.git /content/Ai-video-free

In [ ]:
%cd /content/Ai-video-free
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('Warning: no GPU detected; training may be very slow.')

## Optional: upload your own training JSONL
Each line must be either a messages object or a prompt/completion object. If you skip upload, the tiny sample dataset is used only to verify the pipeline.

In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    import os
    name = next(iter(uploaded))
    if not name.lower().endswith('.jsonl'):
        raise ValueError('Please upload a .jsonl file')
    os.makedirs('/content/mj-ai-data', exist_ok=True)
    raw_path = '/content/mj-ai-data/raw.jsonl'
    with open(raw_path, 'wb') as f: f.write(uploaded[name])
else:
    raw_path = '/content/Ai-video-free/training/seed_data.jsonl'
print('Training data:', raw_path)

In [ ]:
!python /content/Ai-video-free/training/prepare_dataset.py "$raw_path" /content/mj-ai-data/processed.jsonl
!wc -l /content/mj-ai-data/processed.jsonl

## Train the MJ AI adapter
This may take time and consume runtime memory. Three epochs on the tiny starter data will not produce a strong assistant; use your own reviewed dataset for meaningful customization.

In [ ]:
!python /content/Ai-video-free/training/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --data /content/mj-ai-data/processed.jsonl --output /content/mj-ai-lora --epochs 3 --max-length 512

In [ ]:
!cd /content && zip -qr mj-ai-lora.zip mj-ai-lora
from google.colab import files
files.download('/content/mj-ai-lora.zip')

## After training
The downloaded ZIP is the adapter, not a complete standalone base model. To chat with it, download/extract it on a machine that can load the base model, then run:

python model/mj_ai_chat.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter /path/to/mj-ai-lora

Review outputs before sharing. Check the base model license, keep private data out of training sets, and don't claim this tiny first run matches Claude/Gemini.